# BERTurk fine-tuning on Colab (thin shell)

This notebook only installs, clones and calls `python -m tcls.train`. All logic lives in `src/tcls/`.

1. *Runtime → Change runtime type → T4 GPU*.
2. Set `REPO_URL` below.
3. Run all. Run JSONs are written straight to Google Drive, so a disconnected session can be
   resumed: rerun everything, and finished runs are skipped (`--skip-existing`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # <- set this
BRANCH = "main"
CONFIGS = [
    "configs/news_berturk.yaml",
    "configs/news_berturk_len256.yaml",
    "configs/news_berturk_len512.yaml",
    "configs/news_berturk_len512_headtail.yaml",
    "configs/sentiment_berturk.yaml",
    "configs/sentiment_berturk_weighted.yaml",
]
SEEDS = [42, 43, 44]
DRIVE_ROOT = "/content/drive/MyDrive/tcls"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"  # checkpoints: outside the repo
REPORT_DIR = f"{DRIVE_ROOT}/reports"  # per-run JSONs
assert "<your-user>" not in REPO_URL, "Set REPO_URL first"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
import os

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf repo && git clone --depth 1 --branch {BRANCH} {REPO_URL} repo
%cd /content/repo
!pip install -q -e ".[train]"

Optional: `HF_TOKEN` (free, only needed to push models to the Hub later). Add it under the
key icon (*Secrets*) as `HF_TOKEN`. It is read into the environment and never printed.

In [ ]:
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    print("No HF_TOKEN secret (fine: downloads do not need it)")

In [ ]:
# Rebuild the leakage-free splits (same code and seed as the repo; writes data/processed/).
!python -m tcls.data --config configs/news_berturk.yaml configs/sentiment_berturk.yaml --no-tokens --summary /content/data_summary_colab.md

In [ ]:
# Full runs. Each call fine-tunes one (config, seed) and writes one JSON to REPORT_DIR.
for cfg in CONFIGS:
    for seed in SEEDS:
        !python -m tcls.train --config {cfg} --seed {seed} --output-dir {CKPT_DIR} --report-dir {REPORT_DIR} --skip-existing

In [ ]:
# Copy the JSONs into the repo checkout and aggregate (mean ± std over seeds vs. baseline).
!cp {REPORT_DIR}/berturk_*.json reports/
!python -m tcls.aggregate --configs {" ".join(CONFIGS)} --out {REPORT_DIR}/berturk_summary
!cat {REPORT_DIR}/berturk_summary.md

Download `berturk_*.json` and `berturk_summary.{md,json}` from `DRIVE_ROOT/reports` into the
repo's `reports/` and commit them. Checkpoints stay on Drive.